# KIN v2 — Zero-Auth Free Colab Training

Trains **KIN v2** (Qwen2.5-3B cybersecurity LoRA) on the fold-verified 2,512-pair union corpus, pulled entirely from the **public training kit** — no GitHub PAT, no HF token until the optional final upload.

**Requirements:** a Google account (for Drive) and a free Colab T4 GPU. Nothing else.

**Run:** Runtime → Change runtime type → **T4 GPU** → Run all. Hard asserts stop the run on any mismatch.

In [ ]:
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout
print(out)
low = out.lower()
assert any(g in low for g in ['t4', 'l4', 'a100', 'v100']), 'GATE FAILED: no GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.'
print('OK: GPU present')

## 0 — Mount Google Drive (checkpoint destination)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_OUT = '/content/drive/MyDrive/kin-v2-out'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('OK: output dir =', DRIVE_OUT)

## 1 — Fetch the public training kit (zero auth)

One prebuilt corpus file (gzip+base64) + the four pipeline scripts, straight from the public kit repo. No clone, no tokens.

In [ ]:
KIT = 'https://raw.githubusercontent.com/NyxSpecter4/NyxSpecter4/main/kin-v2-train-kit'
!wget -q {KIT}/data/dpo-v2-train.jsonl.gz.b64 -O dpo-v2-train.jsonl.gz.b64
!base64 -d dpo-v2-train.jsonl.gz.b64 | gunzip > dpo-v2-train.jsonl
n = sum(1 for line in open('dpo-v2-train.jsonl') if line.strip())
assert n == 2512, f'GATE FAILED: expected 2512 union DPO pairs, got {n}'
print(f'OK: {n} union DPO pairs (1,637 canonical + 875 staged unique, fold-verified twice)')

In [ ]:
!mkdir -p scripts
!for s in fold-dpo-batches.py prepare_kin_data.py train_kin_v2.py eval_kin.py; do wget -q {KIT}/scripts/$s -O scripts/$s; done
expected = {'fold-dpo-batches.py': 9455, 'prepare_kin_data.py': 3819, 'train_kin_v2.py': 6200, 'eval_kin.py': 5736}
for name, size in expected.items():
    actual = os.path.getsize(f'scripts/{name}')
    assert actual == size, f'GATE FAILED: {name} is {actual} bytes, expected {size}'
print('OK: all 4 pipeline scripts fetched and byte-gated')

In [ ]:
!python scripts/prepare_kin_data.py --input dpo-v2-train.jsonl --output kin-v2-sft.jsonl
n = sum(1 for line in open('kin-v2-sft.jsonl') if line.strip())
assert n == 1171, f'GATE FAILED: expected 1171 SFT rows, got {n}'
print(f'OK: {n} SFT rows ready for training')

## 2 — Install unsloth (~2 min)

In [ ]:
!pip install -q unsloth

## 3 — Train (T4: expect a few hours — keep the tab open)

Qwen2.5-3B · LoRA r16 α32 all-linear · 4-bit · 3 epochs · seq 4096 · T4 fp16 fallback built in. Merged model + adapter are saved to Drive.

In [ ]:
!python scripts/train_kin_v2.py --data kin-v2-sft.jsonl --output-dir {DRIVE_OUT}

## 4 — (Optional) Upload v2 to Hugging Face

Only after the eval gate below **passes**. Needs an HF write token — the single optional credential in the whole pipeline.

In [ ]:
# OPTIONAL — do not run until the eval gate passes.
# from huggingface_hub import HfApi, login
# login()  # paste an HF write token when prompted
# api = HfApi()
# api.create_repo('nyxspecter4/kin-cybersecurity-3b-v2', exist_ok=True)
# api.upload_folder(folder_path=DRIVE_OUT + '/merged', repo_id='nyxspecter4/kin-cybersecurity-3b-v2')

## 5 — Eval gate

**Publish gate: v2 must beat base `Qwen/Qwen2.5-3B-Instruct`.** v1 scored 40/60 vs base 47/60 (Oct 12, 2026). Do not publish below base.

In [ ]:
!python scripts/eval_kin.py --v1 nyxspecter4/kin-cybersecurity-3b --v2 {DRIVE_OUT}/merged --base Qwen/Qwen2.5-3B-Instruct --report gap-delta-report.md
print(open('gap-delta-report.md').read())
print('\nPUBLISH GATE: v2 must score above base before any upload.')